2025/11/4
mlflowをngrokのトンネリングを用いてgoogle colabから接続する方法
方法策定はCursorにて実行。
cursorのAgentタイトル：Google Colabでngrok経由のmlflow使用

In [7]:
!pip -q install mlflow==2.16.2 pyngrok==7.2.1

In [8]:
from pyngrok import ngrok
NGROK_AUTHTOKEN = "1wvkoOmDIsa51PzNifD0Ho1XtCy_87DHFX66mnoB1ZKJSuqXG"  #
ngrok.set_auth_token(NGROK_AUTHTOKEN)

In [9]:
import subprocess, os, time, signal, psutil

# 既存の5000番プロセスがあれば掃除
for p in psutil.process_iter(attrs=["pid","name","cmdline"]):
    try:
        if p.info["cmdline"] and "mlflow" in " ".join(p.info["cmdline"]):
            p.kill()
    except Exception:
        pass

os.makedirs("/content/mlruns", exist_ok=True)
mlflow_cmd = [
    "mlflow", "server",
    "--host", "0.0.0.0",
    "--port", "5000",
    "--backend-store-uri", "sqlite:////content/mlflow.db",
    "--default-artifact-root", "file:///content/mlruns"
]
mlflow_proc = subprocess.Popen(mlflow_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(3)
print("MLflow server started (port 5000)")

MLflow server started (port 5000)


In [10]:
public_tunnel = ngrok.connect(5000)  # http トンネル
public_url = public_tunnel.public_url
public_url

'https://24f1-34-75-156-204.ngrok-free.app'

In [11]:
import mlflow

mlflow.set_tracking_uri(public_url)  # ngrok経由のURLを使用
mlflow.set_experiment("colab-demo")

with mlflow.start_run(run_name="hello-ngrok"):
    mlflow.log_param("alpha", 0.1)
    mlflow.log_metric("rmse", 0.987)

2026/04/25 15:11:14 INFO mlflow.tracking._tracking_service.client: 🏃 View run hello-ngrok at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1/runs/99cd2775d5e74feeb0ca4c9dc083b911.
2026/04/25 15:11:14 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1.
